In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_pry")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyectoyr1")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/countries.csv"

In [0]:
countries_schema = StructType(fields=[
                    StructField("CountryID", IntegerType(), False),
                    StructField("CountryName", StringType(), True),
                    StructField("CountryCode", StringType(), True),
                    StructField("ingestion_date", TimestampType(), True)
])

In [0]:
countries_df = spark.read \
            .option("header", True) \
            .schema(countries_schema) \
            .csv(ruta)

In [0]:
countries_final_df = countries_df.withColumn("ingestion_date", current_timestamp())

In [0]:
countries_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.countries")